# FlyWire Codex Qualification Challenge
## Trial 1

**Phases in this notebook:**
1. Project Setup — path detection, validation, utilities
2. Phase 1 — Data loading, graph construction, basic stats
3. Phase 2 — Exploratory Graph Analysis
4. Phase 3 — Data Quality & Research Planning

---
## 1. Project Setup

In [9]:
# ── Imports ──────────────────────────────────────────────────────────
import sys
import os
import platform
import time
from pathlib import Path
from datetime import datetime

import pandas as pd
import numpy as np
import networkx as nx
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

print(f"Python  : {sys.version}")
print(f"OS      : {platform.system()} {platform.release()}")
print(f"pandas  : {pd.__version__}")
print(f"numpy   : {np.__version__}")
print(f"networkx: {nx.__version__}")
print(f"Time    : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

Python  : 3.12.3 (main, Mar 23 2026, 19:04:32) [GCC 13.3.0]
OS      : Linux 6.14.0-33-generic
pandas  : 3.0.3
numpy   : 2.4.6
networkx: 3.6.1
Time    : 2026-06-07 15:08:09


### 1.1 Path Detection & Directory Setup

In [10]:
def detect_workspace_root() -> Path:
    """Detect workspace root. Works from project root, trial1/, or trial1/notebooks/."""
    markers = ["setup_project.py", "requirements.txt", "README.md"]
    cwd = Path.cwd().resolve()
    for candidate in [cwd, cwd.parent, cwd.parent.parent]:
        if any((candidate / m).exists() for m in markers):
            return candidate
    raise FileNotFoundError("Cannot detect workspace root. Launch from project directory.")

# ── Core path variables ────────────────────────────────────────────
WORKSPACE_ROOT = detect_workspace_root()
TRIAL_ROOT     = WORKSPACE_ROOT / "trial1"
DATA_DIR       = TRIAL_ROOT / "data"
NOTEBOOK_DIR   = TRIAL_ROOT / "notebooks"
RESULTS_DIR    = TRIAL_ROOT / "results"
FIGURES_DIR    = RESULTS_DIR / "figures"
TABLES_DIR     = RESULTS_DIR / "tables"
EXPORTS_DIR    = RESULTS_DIR / "exports"
REPORT_DIR     = TRIAL_ROOT / "report"

# Create all directories
for d in [DATA_DIR, NOTEBOOK_DIR, FIGURES_DIR, TABLES_DIR, EXPORTS_DIR, REPORT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"Working directory : {Path.cwd()}")
print(f"Workspace root    : {WORKSPACE_ROOT}")
print(f"Trial root        : {TRIAL_ROOT}")
print(f"Data directory    : {DATA_DIR}")

Working directory : /home/surjit/Desktop/FlyWire challenge/trial1/notebooks
Workspace root    : /home/surjit/Desktop/FlyWire challenge
Trial root        : /home/surjit/Desktop/FlyWire challenge/trial1
Data directory    : /home/surjit/Desktop/FlyWire challenge/trial1/data


### 1.2 Dataset Availability Check

In [11]:
# ============================================================
# DATASET DISCOVERY & VALIDATION
# ============================================================

from pathlib import Path

# Actual FlyWire dataset filenames
DATASET_FILES = {
    "BANC": "banc_626_edge_list.csv",
    "FAFB": "fafb_783_edge_list.csv",
    "MANC": "manc_1.2.1_edge_list.csv",
    "MAOL": "maol_1.1_edge_list.csv",
    "MCNS": "mcns_0.9_edge_list.csv"
}

print("\n" + "=" * 70)
print("FLYWIRE DATASET AVAILABILITY CHECK")
print("=" * 70)

all_present = True
total_size_bytes = 0

for dataset_name, filename in DATASET_FILES.items():

    file_path = DATA_DIR / filename

    exists = file_path.exists()

    if exists:
        size_mb = file_path.stat().st_size / (1024 * 1024)
        total_size_bytes += file_path.stat().st_size

        print(
            f"✓ {dataset_name:<6} | "
            f"{filename:<30} | "
            f"{size_mb:>8.2f} MB"
        )

    else:
        print(
            f"✗ {dataset_name:<6} | "
            f"{filename:<30} | "
            f"MISSING"
        )
        all_present = False

print("-" * 70)
print(f"Total Dataset Size : {total_size_bytes / (1024**3):.2f} GB")

if all_present:
    print("\n✓ All FlyWire datasets detected successfully.")
else:
    print("\n[WARNING]")
    print("One or more datasets are missing.")
    print(f"Expected location: {DATA_DIR}")

print("=" * 70)


# ============================================================
# PHASE 0 - DATASET PROFILING
# ============================================================

import pandas as pd
from tqdm import tqdm

print("\n")
print("=" * 70)
print("PHASE 0 - DATASET PROFILING")
print("=" * 70)

profiling_results = []

for dataset_name, filename in DATASET_FILES.items():

    file_path = DATA_DIR / filename

    if not file_path.exists():
        continue

    print(f"\nProfiling {dataset_name}...")
    print(f"File: {filename}")

    row_count = 0
    source_nodes = set()
    target_nodes = set()

    chunk_size = 500_000

    try:

        for chunk in tqdm(
            pd.read_csv(file_path, chunksize=chunk_size),
            desc=f"{dataset_name}"
        ):

            row_count += len(chunk)

            source_nodes.update(chunk.iloc[:, 0].unique())
            target_nodes.update(chunk.iloc[:, 1].unique())

        estimated_nodes = len(source_nodes.union(target_nodes))

        profiling_results.append({
            "Dataset": dataset_name,
            "Rows": row_count,
            "Unique Sources": len(source_nodes),
            "Unique Targets": len(target_nodes),
            "Estimated Nodes": estimated_nodes
        })

        print(f"Rows             : {row_count:,}")
        print(f"Unique Sources   : {len(source_nodes):,}")
        print(f"Unique Targets   : {len(target_nodes):,}")
        print(f"Estimated Nodes  : {estimated_nodes:,}")

    except Exception as e:
        print(f"Error profiling {dataset_name}: {e}")

print("\n")
print("=" * 70)
print("DATASET PROFILE SUMMARY")
print("=" * 70)

profile_df = pd.DataFrame(profiling_results)

if not profile_df.empty:
    display(profile_df)

    export_path = EXPORTS_DIR / "dataset_profile_summary.csv"
    profile_df.to_csv(export_path, index=False)

    print(f"\nSaved: {export_path}")

print("=" * 70)


FLYWIRE DATASET AVAILABILITY CHECK
✓ BANC   | banc_626_edge_list.csv         |    99.55 MB
✓ FAFB   | fafb_783_edge_list.csv         |   138.82 MB
✓ MANC   | manc_1.2.1_edge_list.csv       |    66.90 MB
✓ MAOL   | maol_1.1_edge_list.csv         |    83.10 MB
✓ MCNS   | mcns_0.9_edge_list.csv         |    82.14 MB
----------------------------------------------------------------------
Total Dataset Size : 0.46 GB

✓ All FlyWire datasets detected successfully.


PHASE 0 - DATASET PROFILING

Profiling BANC...
File: banc_626_edge_list.csv


BANC: 6it [00:00,  7.48it/s]


Rows             : 2,676,592
Unique Sources   : 108,247
Unique Targets   : 110,092
Estimated Nodes  : 112,885

Profiling FAFB...
File: fafb_783_edge_list.csv


FAFB: 8it [00:01,  7.37it/s]


Rows             : 3,732,460
Unique Sources   : 137,518
Unique Targets   : 130,183
Estimated Nodes  : 138,584

Profiling MANC...
File: manc_1.2.1_edge_list.csv


...

Rows             : 6,484,936
Unique Sources   : 50,870
Unique Targets   : 51,586
Estimated Nodes  : 51,669

Profiling MCNS...
File: mcns_0.9_edge_list.csv


MCNS: 13it [00:00, 13.63it/s]

Rows             : 6,239,112
Unique Sources   : 162,155
Unique Targets   : 161,686
Estimated Nodes  : 165,820


DATASET PROFILE SUMMARY


,Dataset,Rows,Unique Sources,Unique Targets,Estimated Nodes
0,BANC,2676592,108247,110092,112885
1,FAFB,3732460,137518,130183,138584
2,MANC,5305638,23578,23638,23641
3,MAOL,6484936,50870,51586,51669
4,MCNS,6239112,162155,161686,165820



Saved: /home/surjit/Desktop/FlyWire challenge/trial1/results/exports/dataset_profile_summary.csv


### 1.3 Utility Functions

In [12]:
def save_figure(fig, name: str, dpi: int = 150, fmt: str = "png") -> Path:
    """Save a matplotlib figure to FIGURES_DIR."""
    path = FIGURES_DIR / f"{name}.{fmt}"
    fig.savefig(path, dpi=dpi, bbox_inches="tight", facecolor="white")
    print(f"  [SAVED] Figure → {path.relative_to(WORKSPACE_ROOT)}")
    return path


def save_table(df: pd.DataFrame, name: str, fmt: str = "csv") -> Path:
    """Save a DataFrame to TABLES_DIR."""
    path = TABLES_DIR / f"{name}.{fmt}"
    if fmt == "csv":
        df.to_csv(path, index=False)
    else:
        df.to_excel(path, index=False)
    print(f"  [SAVED] Table → {path.relative_to(WORKSPACE_ROOT)}")
    return path


def export_csv(df: pd.DataFrame, name: str) -> Path:
    """Export a DataFrame to EXPORTS_DIR."""
    path = EXPORTS_DIR / f"{name}.csv"
    df.to_csv(path, index=False)
    print(f"  [EXPORTED] → {path.relative_to(WORKSPACE_ROOT)}")
    return path


def print_tree(root: Path, prefix: str = "", max_depth: int = 4, _depth: int = 0):
    """Print directory tree, skipping hidden and __pycache__ dirs."""
    if _depth > max_depth:
        return
    children = sorted(
        [c for c in root.iterdir()
         if not c.name.startswith(".") and c.name != "__pycache__"],
        key=lambda p: (p.is_file(), p.name.lower()),
    )
    for i, child in enumerate(children):
        is_last = i == len(children) - 1
        connector = "└── " if is_last else "├── "
        suffix = "/" if child.is_dir() else ""
        print(f"{prefix}{connector}{child.name}{suffix}")
        if child.is_dir():
            ext = "    " if is_last else "│   "
            print_tree(child, prefix + ext, max_depth, _depth + 1)


def validate_project() -> bool:
    """Validate that the core project structure exists."""
    required = [DATA_DIR, NOTEBOOK_DIR, FIGURES_DIR, TABLES_DIR, EXPORTS_DIR, REPORT_DIR]
    ok = True
    for d in required:
        if not d.exists():
            print(f"  [MISSING] {d.relative_to(WORKSPACE_ROOT)}")
            ok = False
    if ok:
        print("  [OK] All required directories present.")
    return ok


print("Utility functions loaded: save_figure, save_table, export_csv, print_tree, validate_project")

Utility functions loaded: save_figure, save_table, export_csv, print_tree, validate_project


### 1.4 Project Validation & Tree

In [13]:
validate_project()
print()
print("Project Tree")
print("=" * 45)
print_tree(WORKSPACE_ROOT)

  [OK] All required directories present.

Project Tree
├── shared/
├── trial1/
│   ├── data/
│   │   ├── banc_626_edge_list.csv
│   │   ├── fafb_783_edge_list.csv
│   │   ├── manc_1.2.1_edge_list.csv
│   │   ├── maol_1.1_edge_list.csv
│   │   └── mcns_0.9_edge_list.csv
│   ├── notebooks/
│   │   └── flywire.ipynb
│   ├── report/
│   └── results/
│       ├── exports/
│       │   ├── anomaly_report.csv
│       │   ├── basic_graph_stats.csv
│       │   ├── component_analysis.csv
│       │   ├── connected_components.csv
│       │   ├── connectivity_stats.csv
│       │   ├── data_quality_report.csv
│       │   ├── dataset_profile_summary.csv
│       │   ├── degree_stats.csv
│       │   ├── edge_directionality.csv
│       │   ├── graph_structure.csv
│       │   ├── graph_summary.csv
│       │   ├── hub_analysis.csv
│       │   ├── hub_anchor_candidates.html
│       │   ├── pagerank_stats.csv
│       │   ├── reciprocal_fingerprint_stats.csv
│       │   ├── reciprocal_fingerprint_stats.html
│ 

---
## 2. Phase 1 — Data Loading & Graph Construction

### 2.1 Load & Validate CSV Edge Lists

In [14]:
def load_and_validate_edgelist(name: str, data_dir: Path) -> dict:
    """
    Load a FlyWire edge list, validate schema, detect issues,
    remove duplicates, and build a NetworkX DiGraph.
    """

    filepath = data_dir / DATASET_FILES[name]

    print(f"\n{'─'*60}")
    print(f"  Loading: {filepath.name}")
    print(f"{'─'*60}")

    t0 = time.perf_counter()

    # ============================================================
    # LOAD CSV
    # ============================================================
    df = pd.read_csv(filepath, dtype=str)

    raw_rows = len(df)

    print(f"  Raw rows loaded       : {raw_rows:,}")

    # ============================================================
    # SCHEMA VALIDATION
    # ============================================================
    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
    )

    print(f"  Detected columns      : {list(df.columns)}")

    source_col = None
    target_col = None

    for col in df.columns:

        if "source" in col:
            source_col = col

        if "target" in col:
            target_col = col

    if source_col is None or target_col is None:
        raise ValueError(
            f"Could not identify source/target columns in "
            f"{filepath.name}. Found columns: {list(df.columns)}"
        )

    # Keep only source/target columns
    df = df[[source_col, target_col]].copy()

    # Rename to standardized names
    df.columns = ["source", "target"]

    print(
        f"  Schema validation     : OK "
        f"({source_col} -> source, "
        f"{target_col} -> target)"
    )

    # ============================================================
    # MISSING VALUES
    # ============================================================
    n_missing = int(df.isna().sum().sum())

    if n_missing > 0:
        df = df.dropna()

    print(f"  Missing values        : {n_missing:,}")

    # ============================================================
    # STRIP WHITESPACE
    # ============================================================
    df["source"] = df["source"].astype(str).str.strip()
    df["target"] = df["target"].astype(str).str.strip()

    # ============================================================
    # MALFORMED ROWS
    # ============================================================
    malformed_mask = (
        df["source"].eq("")
        |
        df["target"].eq("")
    )

    n_malformed = int(malformed_mask.sum())

    if n_malformed > 0:
        df = df[~malformed_mask]

    print(f"  Malformed rows        : {n_malformed:,}")

    # ============================================================
    # DUPLICATE EDGES
    # ============================================================
    before = len(df)

    df = df.drop_duplicates(
        subset=["source", "target"]
    )

    n_duplicates = before - len(df)

    print(f"  Duplicate edges       : {n_duplicates:,} removed")

    # ============================================================
    # BUILD GRAPH
    # ============================================================
    G = nx.from_pandas_edgelist(
        df,
        source="source",
        target="target",
        create_using=nx.DiGraph()
    )

    elapsed = time.perf_counter() - t0

    n_nodes = G.number_of_nodes()
    n_edges = G.number_of_edges()

    print(f"  Nodes                 : {n_nodes:,}")
    print(f"  Edges                 : {n_edges:,}")
    print(f"  Load time             : {elapsed:.2f}s")

    return {
        "name": name,
        "graph": G,
        "nodes": n_nodes,
        "edges": n_edges,
        "duplicates_removed": n_duplicates,
        "missing_values": n_missing,
        "malformed_rows": n_malformed,
        "load_time_s": round(elapsed, 3),
    }

### 2.2 Load All Datasets

In [15]:
graphs = {}
summaries = []

total_t0 = time.perf_counter()

for name, filename in tqdm(
    DATASET_FILES.items(),
    desc="Loading datasets"
):

    filepath = DATA_DIR / filename

    if not filepath.exists():
        print(f"\n[SKIP] {filename} not found in {DATA_DIR}")
        continue

    result = load_and_validate_edgelist(
        name,
        DATA_DIR
    )

    graphs[name] = result["graph"]

    summaries.append({
        "dataset": name,
        "nodes": result["nodes"],
        "edges": result["edges"],
        "duplicates_removed": result["duplicates_removed"],
        "missing_values": result["missing_values"],
        "load_time_s": result["load_time_s"],
    })

total_elapsed = time.perf_counter() - total_t0

print("\n" + "=" * 60)
print(f"All datasets loaded in {total_elapsed:.2f}s")
print(f"Graphs in memory: {list(graphs.keys())}")
print("=" * 60)

Loading datasets:   0%|          | 0/5 [00:00<?, ?it/s]


────────────────────────────────────────────────────────────
  Loading: banc_626_edge_list.csv
────────────────────────────────────────────────────────────
  Raw rows loaded       : 2,676,592
  Detected columns      : ['source neuron id', 'target neuron id']
  Schema validation     : OK (source neuron id -> source, target neuron id -> target)
  Missing values        : 0
  Malformed rows        : 0
  Duplicate edges       : 0 removed


Loading datasets:  20%|██        | 1/5 [00:11<00:47, 11.77s/it]

  Nodes                 : 112,885
  Edges                 : 2,676,592
  Load time             : 11.72s

────────────────────────────────────────────────────────────
  Loading: fafb_783_edge_list.csv
────────────────────────────────────────────────────────────
  Raw rows loaded       : 3,732,460
  Detected columns      : ['source neuron id', 'target neuron id']
  Schema validation     : OK (source neuron id -> source, target neuron id -> target)
  Missing values        : 0
  Malformed rows        : 0
  Duplicate edges       : 0 removed


Loading datasets:  40%|████      | 2/5 [00:26<00:40, 13.40s/it]

...

  Nodes                 : 23,641
  Edges                 : 5,305,638
  Load time             : 15.26s

────────────────────────────────────────────────────────────
  Loading: maol_1.1_edge_list.csv
────────────────────────────────────────────────────────────
  Raw rows loaded       : 6,484,936
  Detected columns      : ['source neuron id', 'target neuron id']
  Schema validation     : OK (source neuron id -> source, target neuron id -> target)
  Missing values        : 0
  Malformed rows        : 0
  Duplicate edges       : 0 removed


Loading datasets:  80%|████████  | 4/5 [01:01<00:16, 16.31s/it]

  Nodes                 : 51,669
  Edges                 : 6,484,936
  Load time             : 19.42s

────────────────────────────────────────────────────────────
  Loading: mcns_0.9_edge_list.csv
────────────────────────────────────────────────────────────
  Raw rows loaded       : 6,239,112
  Detected columns      : ['source neuron id', 'target neuron id']
  Schema validation     : OK (source neuron id -> source, target neuron id -> target)
  Missing values        : 0
  Malformed rows        : 0
  Duplicate edges       : 0 removed


Loading datasets: 100%|██████████| 5/5 [01:20<00:00, 16.05s/it]

  Nodes                 : 165,820
  Edges                 : 6,239,112
  Load time             : 19.13s

All datasets loaded in 80.26s
Graphs in memory: ['BANC', 'FAFB', 'MANC', 'MAOL', 'MCNS']


### 2.3 Summary Table

In [16]:
if summaries:
    df_summary = pd.DataFrame(summaries)
    display(df_summary)
else:
    print("No datasets were loaded. Place CSV files in trial1/data/ and re-run.")

,dataset,nodes,edges,duplicates_removed,missing_values,load_time_s
0,BANC,112885,2676592,0,0,11.720
1,FAFB,138584,3732460,0,0,14.479
2,MANC,23641,5305638,0,0,15.255
3,MAOL,51669,6484936,0,0,19.424
4,MCNS,165820,6239112,0,0,19.129


### 2.4 Export Basic Graph Statistics

In [17]:
if summaries:
    export_csv(df_summary, "basic_graph_stats")
    print("\nDone. Phase 1 complete.")
else:
    print("Nothing to export.")

  [EXPORTED] → trial1/results/exports/basic_graph_stats.csv

Done. Phase 1 complete.


---
# Phase 2 — Exploratory Graph Analysis

This phase performs comprehensive structural analysis of all loaded connectome graphs:
- Basic connectivity statistics
- Connected component analysis
- Degree distributions & percentiles
- PageRank centrality
- Reciprocity analysis
- Graph structure (shortest paths / diameter where feasible)
- Publication-quality visualizations

## 3.1 Basic Connectivity Statistics

Compute node/edge counts, density, and degree statistics for every graph.

In [18]:
t0 = time.perf_counter()

basic_rows = []
for name, G in graphs.items():
    in_deg  = np.array([d for _, d in G.in_degree()])
    out_deg = np.array([d for _, d in G.out_degree()])
    tot_deg = in_deg + out_deg
    n = G.number_of_nodes()
    m = G.number_of_edges()
    basic_rows.append({
        'dataset': name,
        'nodes': n,
        'edges': m,
        'density': nx.density(G),
        'avg_in_degree': in_deg.mean(),
        'avg_out_degree': out_deg.mean(),
        'median_in_degree': float(np.median(in_deg)),
        'median_out_degree': float(np.median(out_deg)),
        'max_in_degree': int(in_deg.max()),
        'max_out_degree': int(out_deg.max()),
        'std_total_degree': tot_deg.std(),
    })

graph_summary_df = pd.DataFrame(basic_rows)
display(graph_summary_df)
export_csv(graph_summary_df, 'connectivity_stats')
print(f'\nCompleted in {time.perf_counter()-t0:.2f}s')

,dataset,nodes,edges,density,avg_in_degree,avg_out_degree,median_in_degree,median_out_degree,max_in_degree,max_out_degree,std_total_degree
0,BANC,112885,2676592,0.000210,23.710785,23.710785,10.0,13.0,1723,1858,79.863049
1,FAFB,138584,3732460,0.000194,26.932835,26.932835,14.0,17.0,6261,6523,108.409045
2,MANC,23641,5305638,0.009493,224.425278,224.425278,173.0,163.0,2486,4752,418.307986
3,MAOL,51669,6484936,0.002429,125.509222,125.509222,79.0,91.0,11496,11215,355.495784
4,MCNS,165820,6239112,0.000227,37.625811,37.625811,19.0,25.0,6660,7570,126.085236


  [EXPORTED] → trial1/results/exports/connectivity_stats.csv

Completed in 0.59s


## 3.2 Connected Component Analysis

Compute weakly and strongly connected components for each graph.

In [19]:
t0 = time.perf_counter()

cc_rows = []
for name, G in graphs.items():
    n = G.number_of_nodes()
    # Weakly connected components
    wccs = list(nx.weakly_connected_components(G))
    largest_wcc = max(len(c) for c in wccs)
    # Strongly connected components
    sccs = list(nx.strongly_connected_components(G))
    largest_scc = max(len(c) for c in sccs)
    cc_rows.append({
        'dataset': name,
        'num_wcc': len(wccs),
        'largest_wcc': largest_wcc,
        'pct_in_largest_wcc': round(100*largest_wcc/n, 2),
        'num_scc': len(sccs),
        'largest_scc': largest_scc,
        'pct_in_largest_scc': round(100*largest_scc/n, 2),
    })
    print(f'{name}: WCC={len(wccs):,} (largest {largest_wcc:,}), SCC={len(sccs):,} (largest {largest_scc:,})')

cc_df = pd.DataFrame(cc_rows)
display(cc_df)
export_csv(cc_df, 'connected_components')
print(f'\nCompleted in {time.perf_counter()-t0:.2f}s')

BANC: WCC=8 (largest 112,871), SCC=7,764 (largest 105,080)
FAFB: WCC=204 (largest 136,997), SCC=10,449 (largest 127,536)
MANC: WCC=1 (largest 23,641), SCC=71 (largest 23,569)
MAOL: WCC=45 (largest 51,503), SCC=929 (largest 50,709)
MCNS: WCC=1 (largest 165,820), SCC=7,985 (largest 157,789)


,dataset,num_wcc,largest_wcc,pct_in_largest_wcc,num_scc,largest_scc,pct_in_largest_scc
0,BANC,8,112871,99.99,7764,105080,93.09
1,FAFB,204,136997,98.85,10449,127536,92.03
2,MANC,1,23641,100.00,71,23569,99.70
3,MAOL,45,51503,99.68,929,50709,98.14
4,MCNS,1,165820,100.00,7985,157789,95.16


  [EXPORTED] → trial1/results/exports/connected_components.csv

Completed in 21.21s


## 3.3 Degree Analysis

### Degree distributions, percentiles, and top-k hub nodes.

In [20]:
t0 = time.perf_counter()

# Store degree arrays for plotting
degree_data = {}
percentile_rows = []
pcts = [50, 75, 90, 95, 99]

for name, G in graphs.items():
    in_deg  = np.array([d for _, d in G.in_degree()])
    out_deg = np.array([d for _, d in G.out_degree()])
    tot_deg = in_deg + out_deg
    degree_data[name] = {'in': in_deg, 'out': out_deg, 'total': tot_deg}
    row = {'dataset': name}
    for p in pcts:
        row[f'in_P{p}']  = int(np.percentile(in_deg, p))
        row[f'out_P{p}'] = int(np.percentile(out_deg, p))
    percentile_rows.append(row)

pct_df = pd.DataFrame(percentile_rows)
display(pct_df)
print(f'\nCompleted in {time.perf_counter()-t0:.2f}s')

,dataset,in_P50,out_P50,in_P75,out_P75,in_P90,out_P90,in_P95,out_P95,in_P99,out_P99
0,BANC,10,13,25,26,58,53,91,83,205,188
1,FAFB,14,17,26,31,59,54,96,75,215,170
2,MANC,173,163,309,271,492,440,635,596,986,1059
3,MAOL,79,91,153,144,241,217,334,293,736,693
4,MCNS,19,25,39,43,87,72,132,107,279,227



Completed in 0.37s


### Top-20 Hub Nodes (In-Degree & Out-Degree)

In [21]:
top_k = 20
hub_frames = []

for name, G in graphs.items():
    # Top in-degree
    in_sorted = sorted(G.in_degree(), key=lambda x: x[1], reverse=True)[:top_k]
    df_in = pd.DataFrame(in_sorted, columns=['node','in_degree'])
    df_in.insert(0, 'dataset', name)
    df_in['rank'] = range(1, top_k+1)
    # Top out-degree
    out_sorted = sorted(G.out_degree(), key=lambda x: x[1], reverse=True)[:top_k]
    df_out = pd.DataFrame(out_sorted, columns=['node','out_degree'])
    df_out.insert(0, 'dataset', name)
    df_out['rank'] = range(1, top_k+1)
    hub_frames.extend([df_in, df_out])
    print(f'\n{name} — Top-3 in-degree: {in_sorted[:3]}')
    print(f'{name} — Top-3 out-degree: {out_sorted[:3]}')

# Combine and export
top_in_all = pd.concat([f for i,f in enumerate(hub_frames) if i%2==0], ignore_index=True)
top_out_all = pd.concat([f for i,f in enumerate(hub_frames) if i%2==1], ignore_index=True)
export_csv(top_in_all, 'top20_in_degree')
export_csv(top_out_all, 'top20_out_degree')


BANC — Top-3 in-degree: [('720575941688977932', 1723), ('720575941526929485', 1705), ('720575941633862331', 1586)]
BANC — Top-3 out-degree: [('720575941539358613', 1858), ('720575941633862331', 1844), ('720575941612297958', 1420)]

FAFB — Top-3 in-degree: [('720575940626979621', 6261), ('720575940628908548', 5997), ('720575940613583001', 2883)]
FAFB — Top-3 out-degree: [('720575940626979621', 6523), ('720575940628908548', 6447), ('720575940613583001', 2795)]

MANC — Top-3 in-degree: [('10311', 2486), ('10477', 2397), ('12749', 2372)]
MANC — Top-3 out-degree: [('10311', 4752), ('10477', 4698), ('10222', 4578)]

MAOL — Top-3 in-degree: [('10009', 11496), ('10051', 6195), ('10330', 6175)]
MAOL — Top-3 out-degree: [('10093', 11215), ('10046', 10883), ('10072', 10421)]

MCNS — Top-3 in-degree: [('10009', 6660), ('10157', 5668), ('10085', 2582)]
MCNS — Top-3 out-degree: [('10009', 7570), ('10157', 7209), ('10503', 2743)]
  [EXPORTED] → trial1/results/exports/top20_in_degree.csv
  [EXPORTED]

PosixPath('/home/surjit/Desktop/FlyWire challenge/trial1/results/exports/top20_out_degree.csv')

### Degree Distribution Plots

In [22]:
sns.set_style('whitegrid')

# ── In-degree & Out-degree histograms ──
fig, axes = plt.subplots(len(graphs), 2, figsize=(14, 4*len(graphs)))
if len(graphs) == 1:
    axes = axes.reshape(1, -1)

for i, (name, dd) in enumerate(degree_data.items()):
    axes[i,0].hist(dd['in'], bins=50, color='steelblue', edgecolor='white', alpha=0.85)
    axes[i,0].set_title(f'{name} — In-Degree', fontsize=12)
    axes[i,0].set_xlabel('In-degree')
    axes[i,0].set_ylabel('Count')
    axes[i,1].hist(dd['out'], bins=50, color='coral', edgecolor='white', alpha=0.85)
    axes[i,1].set_title(f'{name} — Out-Degree', fontsize=12)
    axes[i,1].set_xlabel('Out-degree')
    axes[i,1].set_ylabel('Count')

fig.tight_layout()
save_figure(fig, 'degree_histograms')
plt.show()

  [SAVED] Figure → trial1/results/figures/degree_histograms.png


/tmp/ipykernel_5534/2984333024.py:20: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Log-Scale Degree Distributions

In [23]:
fig, axes = plt.subplots(1, len(graphs), figsize=(5*len(graphs), 4))
if len(graphs) == 1:
    axes = [axes]

for ax, (name, dd) in zip(axes, degree_data.items()):
    vals, counts = np.unique(dd['total'], return_counts=True)
    ax.scatter(vals, counts, s=8, alpha=0.6, color='darkslateblue')
    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.set_title(f'{name} — Total Degree (log-log)', fontsize=11)
    ax.set_xlabel('Degree')
    ax.set_ylabel('Frequency')

fig.tight_layout()
save_figure(fig, 'degree_loglog')
plt.show()

  [SAVED] Figure → trial1/results/figures/degree_loglog.png


/tmp/ipykernel_5534/685373613.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Complementary Cumulative Distribution (CCDF)

In [24]:
fig, ax = plt.subplots(figsize=(8, 5))
colors = plt.cm.Set2(np.linspace(0, 1, len(graphs)))

for (name, dd), c in zip(degree_data.items(), colors):
    sorted_deg = np.sort(dd['total'])[::-1]
    ccdf = np.arange(1, len(sorted_deg)+1) / len(sorted_deg)
    ax.plot(sorted_deg, ccdf, label=name, color=c, linewidth=1.5)

ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('Degree')
ax.set_ylabel('CCDF')
ax.set_title('Total Degree — CCDF Comparison')
ax.legend()
fig.tight_layout()
save_figure(fig, 'degree_ccdf')
plt.show()

  [SAVED] Figure → trial1/results/figures/degree_ccdf.png


/tmp/ipykernel_5534/1879538263.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [25]:
export_csv(pct_df, 'degree_stats')
print('Degree analysis complete.')

  [EXPORTED] → trial1/results/exports/degree_stats.csv
Degree analysis complete.


## 3.4 PageRank Analysis

Compute PageRank centrality and identify the most influential nodes.

In [26]:
t0 = time.perf_counter()
pagerank_scores = {}
pr_rows = []

for name, G in tqdm(graphs.items(), desc='PageRank'):
    pr = nx.pagerank(G, max_iter=100, tol=1e-06)
    pagerank_scores[name] = pr
    vals = np.array(list(pr.values()))
    pr_rows.append({
        'dataset': name,
        'mean_pr': vals.mean(),
        'std_pr': vals.std(),
        'max_pr': vals.max(),
        'min_pr': vals.min(),
    })

pr_summary = pd.DataFrame(pr_rows)
display(pr_summary)
export_csv(pr_summary, 'pagerank_stats')
print(f'\nCompleted in {time.perf_counter()-t0:.2f}s')

PageRank: 100%|██████████| 5/5 [00:23<00:00,  4.66s/it]


,dataset,mean_pr,std_pr,max_pr,min_pr
0,BANC,0.000009,0.000021,0.002620,1.459447e-06
1,FAFB,0.000007,0.000016,0.001129,1.108727e-06
2,MANC,0.000042,0.000035,0.000484,6.420495e-06
3,MAOL,0.000019,0.000028,0.001212,3.077677e-06
4,MCNS,0.000006,0.000011,0.000758,9.881339e-07


  [EXPORTED] → trial1/results/exports/pagerank_stats.csv

Completed in 23.29s


### Top-20 PageRank Nodes

In [27]:
top_pr_frames = []
for name, pr in pagerank_scores.items():
    top = sorted(pr.items(), key=lambda x: x[1], reverse=True)[:20]
    df_pr = pd.DataFrame(top, columns=['node', 'pagerank'])
    df_pr.insert(0, 'dataset', name)
    df_pr['rank'] = range(1, 21)
    top_pr_frames.append(df_pr)
    print(f'{name} — Top-3 PageRank: {top[:3]}')

top_pr_all = pd.concat(top_pr_frames, ignore_index=True)
export_csv(top_pr_all, 'top20_pagerank')

BANC — Top-3 PageRank: [('720575941539358613', 0.002619757152041062), ('720575941633862331', 0.0022019835793681284), ('720575941688977932', 0.0008748667444460389)]
FAFB — Top-3 PageRank: [('720575940628908548', 0.0011288259591334094), ('720575940626979621', 0.0009088486252627922), ('720575940613583001', 0.0007713978801940416)]
MANC — Top-3 PageRank: [('10477', 0.00048435787228153215), ('10330', 0.0003903477898164752), ('10222', 0.00039025492151715526)]
MAOL — Top-3 PageRank: [('10009', 0.0012122954504573131), ('10051', 0.0011995298002094234), ('10302', 0.0008064659490188985)]
MCNS — Top-3 PageRank: [('10157', 0.0007581527118752559), ('10009', 0.000674579985856955), ('10085', 0.0005583591579714393)]
  [EXPORTED] → trial1/results/exports/top20_pagerank.csv


PosixPath('/home/surjit/Desktop/FlyWire challenge/trial1/results/exports/top20_pagerank.csv')

## 3.5 Reciprocity Analysis

Measure the fraction of edges that have a reciprocal (reverse) edge.

In [28]:
t0 = time.perf_counter()
recip_rows = []

for name, G in graphs.items():
    edges = set(G.edges())
    reciprocal = sum(1 for u, v in edges if (v, u) in edges)
    ratio = reciprocal / len(edges) if len(edges) > 0 else 0
    recip_rows.append({
        'dataset': name,
        'total_edges': len(edges),
        'reciprocal_edges': reciprocal,
        'reciprocity_ratio': round(ratio, 4),
    })
    print(f'{name}: {reciprocal:,} reciprocal / {len(edges):,} total = {ratio:.4f}')

recip_df = pd.DataFrame(recip_rows)
display(recip_df)
export_csv(recip_df, 'reciprocity')
print(f'\nCompleted in {time.perf_counter()-t0:.2f}s')

BANC: 383,424 reciprocal / 2,676,592 total = 0.1433
FAFB: 620,180 reciprocal / 3,732,460 total = 0.1662
MANC: 1,612,184 reciprocal / 5,305,638 total = 0.3039
MAOL: 1,826,775 reciprocal / 6,484,936 total = 0.2817
MCNS: 977,866 reciprocal / 6,239,112 total = 0.1567


,dataset,total_edges,reciprocal_edges,reciprocity_ratio
0,BANC,2676592,383424,0.1433
1,FAFB,3732460,620180,0.1662
2,MANC,5305638,1612184,0.3039
3,MAOL,6484936,1826775,0.2817
4,MCNS,6239112,977866,0.1567


  [EXPORTED] → trial1/results/exports/reciprocity.csv

Completed in 18.76s


## 3.6 Graph Structure — Shortest Paths & Diameter

Computed **only** for graphs with ≤ 10,000 nodes (otherwise prohibitively expensive).

In [29]:
SIZE_THRESHOLD = 10_000
t0 = time.perf_counter()
struct_rows = []

for name, G in graphs.items():
    n = G.number_of_nodes()
    if n > SIZE_THRESHOLD:
        print(f'{name}: {n:,} nodes > {SIZE_THRESHOLD:,} threshold — SKIPPED (O(n²) cost)')
        struct_rows.append({'dataset': name, 'avg_shortest_path': None, 'diameter': None, 'note': 'skipped — too large'})
        continue
    if not nx.is_weakly_connected(G):
        largest_wcc_nodes = max(nx.weakly_connected_components(G), key=len)
        Gsub = G.subgraph(largest_wcc_nodes).copy()
        print(f'{name}: using largest WCC ({Gsub.number_of_nodes():,} nodes)')
    else:
        Gsub = G
    try:
        avg_sp = nx.average_shortest_path_length(Gsub)
        diam = nx.diameter(Gsub)
        print(f'{name}: avg_sp={avg_sp:.3f}, diameter={diam}')
        struct_rows.append({'dataset': name, 'avg_shortest_path': round(avg_sp,3), 'diameter': diam, 'note': 'computed'})
    except Exception as e:
        print(f'{name}: could not compute — {e}')
        struct_rows.append({'dataset': name, 'avg_shortest_path': None, 'diameter': None, 'note': str(e)})

struct_df = pd.DataFrame(struct_rows)
display(struct_df)
export_csv(struct_df, 'graph_structure')
print(f'\nCompleted in {time.perf_counter()-t0:.2f}s')

BANC: 112,885 nodes > 10,000 threshold — SKIPPED (O(n²) cost)
FAFB: 138,584 nodes > 10,000 threshold — SKIPPED (O(n²) cost)
MANC: 23,641 nodes > 10,000 threshold — SKIPPED (O(n²) cost)
MAOL: 51,669 nodes > 10,000 threshold — SKIPPED (O(n²) cost)
MCNS: 165,820 nodes > 10,000 threshold — SKIPPED (O(n²) cost)


,dataset,avg_shortest_path,diameter,note
0,BANC,None,None,skipped — too large
1,FAFB,None,None,skipped — too large
2,MANC,None,None,skipped — too large
3,MAOL,None,None,skipped — too large
4,MCNS,None,None,skipped — too large


  [EXPORTED] → trial1/results/exports/graph_structure.csv

Completed in 0.01s


## 3.7 Visualization Suite

Publication-quality comparative bar charts across all datasets.

In [30]:
sns.set_style('whitegrid')
palette = sns.color_palette('Set2', len(graphs))
names = list(graphs.keys())

def bar_chart(values, ylabel, title, filename, color_list=None):
    fig, ax = plt.subplots(figsize=(8, 4))
    c = color_list if color_list else palette
    bars = ax.bar(names, values, color=c, edgecolor='white', width=0.6)
    ax.set_ylabel(ylabel)
    ax.set_title(title, fontsize=13, fontweight='bold')
    for b, v in zip(bars, values):
        fmt = f'{v:,.0f}' if v > 100 else f'{v:.4f}'
        ax.text(b.get_x()+b.get_width()/2, b.get_height(), fmt,
                ha='center', va='bottom', fontsize=9)
    fig.tight_layout()
    save_figure(fig, filename)
    plt.show()

# 1. Nodes
bar_chart([G.number_of_nodes() for G in graphs.values()], 'Nodes', 'Nodes per Dataset', 'nodes_per_dataset')
# 2. Edges
bar_chart([G.number_of_edges() for G in graphs.values()], 'Edges', 'Edges per Dataset', 'edges_per_dataset')
# 3. Density
bar_chart(graph_summary_df['density'].tolist(), 'Density', 'Graph Density Comparison', 'density_comparison')
# 4. Largest SCC
bar_chart(cc_df['largest_scc'].tolist(), 'Nodes in SCC', 'Largest Strongly Connected Component', 'largest_scc')
# 5. Largest WCC
bar_chart(cc_df['largest_wcc'].tolist(), 'Nodes in WCC', 'Largest Weakly Connected Component', 'largest_wcc')
# 6. Reciprocity
bar_chart(recip_df['reciprocity_ratio'].tolist(), 'Ratio', 'Reciprocity Comparison', 'reciprocity_comparison')

  [SAVED] Figure → trial1/results/figures/nodes_per_dataset.png


/tmp/ipykernel_5534/2455548680.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


  [SAVED] Figure → trial1/results/figures/edges_per_dataset.png
  [SAVED] Figure → trial1/results/figures/density_comparison.png
  [SAVED] Figure → trial1/results/figures/largest_scc.png
  [SAVED] Figure → trial1/results/figures/largest_wcc.png
  [SAVED] Figure → trial1/results/figures/reciprocity_comparison.png


### PageRank Distribution Comparison

In [31]:
fig, ax = plt.subplots(figsize=(8, 5))
colors = plt.cm.Set2(np.linspace(0, 1, len(graphs)))

for (name, pr), c in zip(pagerank_scores.items(), colors):
    vals = sorted(pr.values(), reverse=True)
    ax.plot(range(1, len(vals)+1), vals, label=name, color=c, linewidth=1.2)

ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('Rank')
ax.set_ylabel('PageRank Score')
ax.set_title('PageRank Distribution (log-log)')
ax.legend()
fig.tight_layout()
save_figure(fig, 'pagerank_distributions')
plt.show()

  [SAVED] Figure → trial1/results/figures/pagerank_distributions.png


/tmp/ipykernel_5534/1984309495.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3.8 Combined Graph Summary Export

In [32]:
# Merge all per-dataset stats into one master table
master = graph_summary_df.merge(cc_df, on='dataset').merge(recip_df[['dataset','reciprocity_ratio']], on='dataset')
display(master)
export_csv(master, 'graph_summary')
print('\n' + '='*60)
print('  Phase 2 — Exploratory Graph Analysis COMPLETE')
print('='*60)

,dataset,nodes,edges,density,avg_in_degree,avg_out_degree,median_in_degree,median_out_degree,max_in_degree,max_out_degree,std_total_degree,num_wcc,largest_wcc,pct_in_largest_wcc,num_scc,largest_scc,pct_in_largest_scc,reciprocity_ratio
0,BANC,112885,2676592,0.000210,23.710785,23.710785,10.0,13.0,1723,1858,79.863049,8,112871,99.99,7764,105080,93.09,0.1433
1,FAFB,138584,3732460,0.000194,26.932835,26.932835,14.0,17.0,6261,6523,108.409045,204,136997,98.85,10449,127536,92.03,0.1662
2,MANC,23641,5305638,0.009493,224.425278,224.425278,173.0,163.0,2486,4752,418.307986,1,23641,100.00,71,23569,99.70,0.3039
3,MAOL,51669,6484936,0.002429,125.509222,125.509222,79.0,91.0,11496,11215,355.495784,45,51503,99.68,929,50709,98.14,0.2817
4,MCNS,165820,6239112,0.000227,37.625811,37.625811,19.0,25.0,6660,7570,126.085236,1,165820,100.00,7985,157789,95.16,0.1567


  [EXPORTED] → trial1/results/exports/graph_summary.csv

  Phase 2 — Exploratory Graph Analysis COMPLETE


---
# Phase 3 — Data Quality & Sanity Checks

Systematic data-quality audit and research planning based on Stages 1–2.

# Phase 3 Assumptions (Derived from Phase 2)

The following assumptions are established from Phase 2 findings and govern all
Phase 3 analysis and Phase 4 design decisions.

---

### A1 — Giant Component Assumption

**Evidence:**
Largest WCC covers 98.85–100% of nodes.

**Implication:**
Graph decomposition by weak components is not useful.

---

### A2 — Giant SCC Assumption

**Evidence:**
Largest SCC covers 92.03–99.70% of nodes.

**Implication:**
SCC decomposition should not be a Phase 4 strategy.

---

### A3 — Heavy-Tailed Degree Assumption

**Evidence:**
Extreme hub nodes exist across all datasets.

**Implication:**
Hubs should be used as anchor candidates.

---

### A4 — Reciprocity Assumption

**Evidence:**
Reciprocity varies from 0.143–0.304.

**Implication:**
Reciprocity should be incorporated into node fingerprints.

---

### A5 — Dense vs Sparse Family Assumption

**Evidence:**
Datasets separate into sparse and dense families.

**Implication:**
Matching thresholds may require family-specific tuning.

---

### A6 — Exact Matching Infeasibility

**Evidence:**
MCNS contains 165,820 nodes.

**Implication:**
Exact graph matching is computationally infeasible.

---

### A7 — WL Necessity

**Evidence:**
Large graph sizes require candidate refinement.

**Implication:**
WL refinement should be part of Phase 4.

---

### A8 — Hub Seeding Assumption

**Evidence:**
Heavy-tailed degree distributions create highly distinctive hubs.

**Implication:**
Hub-seeded matching is preferred over exhaustive matching.

## 4.1 Self-Loop Analysis

In [33]:
t0 = time.perf_counter()
sl_rows = []
for name, G in graphs.items():
    loops = list(nx.selfloop_edges(G))
    n_loops = len(loops)
    pct = 100 * n_loops / G.number_of_edges() if G.number_of_edges() else 0
    examples = loops[:5]
    sl_rows.append({'dataset':name,'self_loops':n_loops,'pct_of_edges':round(pct,4),'examples':str(examples)})
    print(f'{name}: {n_loops:,} self-loops ({pct:.4f}%)')

sl_df = pd.DataFrame(sl_rows)
display(sl_df)
print(f'Completed in {time.perf_counter()-t0:.2f}s')

BANC: 0 self-loops (0.0000%)
FAFB: 0 self-loops (0.0000%)
MANC: 36 self-loops (0.0007%)
MAOL: 263 self-loops (0.0041%)
MCNS: 18 self-loops (0.0003%)


,dataset,self_loops,pct_of_edges,examples
0,BANC,0,0.0000,[]
1,FAFB,0,0.0000,[]
2,MANC,36,0.0007,"[('11827', '11827'), ('37628', '37628'), ('193..."
3,MAOL,263,0.0041,"[('19070', '19070'), ('33715', '33715'), ('386..."
4,MCNS,18,0.0003,"[('10540', '10540'), ('37975', '37975'), ('492..."


Completed in 0.45s


## 4.2 Duplicate Edge Report (from Stage 1)

In [34]:
# Reuse Stage 1 summary if available
dup_path = EXPORTS_DIR / 'basic_graph_stats.csv'
if dup_path.exists():
    dup_df = pd.read_csv(dup_path)
    if 'duplicates_removed' in dup_df.columns:
        display(dup_df[['dataset','edges','duplicates_removed','missing_values']])
    else:
        print('duplicates_removed column not found in basic_graph_stats.csv')
else:
    print('basic_graph_stats.csv not found — run Phase 1 first.')

,dataset,edges,duplicates_removed,missing_values
0,BANC,2676592,0,0
1,FAFB,3732460,0,0
2,MANC,5305638,0,0
3,MAOL,6484936,0,0
4,MCNS,6239112,0,0


## 4.3 Isolated Node Analysis

Nodes with **in_degree = 0 AND out_degree = 0**.

In [35]:
iso_rows = []
for name, G in graphs.items():
    isolated = [n for n in G.nodes() if G.in_degree(n)==0 and G.out_degree(n)==0]
    iso_rows.append({'dataset':name,'isolated_nodes':len(isolated),'pct':round(100*len(isolated)/G.number_of_nodes(),4) if G.number_of_nodes() else 0})
    print(f'{name}: {len(isolated):,} isolated nodes')

iso_df = pd.DataFrame(iso_rows)
display(iso_df)

BANC: 0 isolated nodes
FAFB: 0 isolated nodes
MANC: 0 isolated nodes
MAOL: 0 isolated nodes
MCNS: 0 isolated nodes


,dataset,isolated_nodes,pct
0,BANC,0,0.0
1,FAFB,0,0.0
2,MANC,0,0.0
3,MAOL,0,0.0
4,MCNS,0,0.0


## 4.4 Source-Only Nodes

Nodes with **out_degree > 0, in_degree = 0**.

In [36]:
src_rows = []
for name, G in graphs.items():
    sources = [n for n in G.nodes() if G.out_degree(n)>0 and G.in_degree(n)==0]
    src_rows.append({'dataset':name,'source_only':len(sources),'pct':round(100*len(sources)/G.number_of_nodes(),4)})
    print(f'{name}: {len(sources):,} source-only nodes')

src_df = pd.DataFrame(src_rows)
display(src_df)

BANC: 2,793 source-only nodes
FAFB: 8,401 source-only nodes
MANC: 3 source-only nodes
MAOL: 83 source-only nodes
MCNS: 4,134 source-only nodes


,dataset,source_only,pct
0,BANC,2793,2.4742
1,FAFB,8401,6.0620
2,MANC,3,0.0127
3,MAOL,83,0.1606
4,MCNS,4134,2.4931


## 4.5 Sink-Only Nodes

Nodes with **in_degree > 0, out_degree = 0**.

In [37]:
sink_rows = []
for name, G in graphs.items():
    sinks = [n for n in G.nodes() if G.in_degree(n)>0 and G.out_degree(n)==0]
    sink_rows.append({'dataset':name,'sink_only':len(sinks),'pct':round(100*len(sinks)/G.number_of_nodes(),4)})
    print(f'{name}: {len(sinks):,} sink-only nodes')

sink_df = pd.DataFrame(sink_rows)
display(sink_df)

BANC: 4,638 sink-only nodes
FAFB: 1,066 sink-only nodes
MANC: 63 sink-only nodes
MAOL: 799 sink-only nodes
MCNS: 3,665 sink-only nodes


,dataset,sink_only,pct
0,BANC,4638,4.1086
1,FAFB,1066,0.7692
2,MANC,63,0.2665
3,MAOL,799,1.5464
4,MCNS,3665,2.2102


## 4.6 SCC Size Distribution

In [38]:
t0 = time.perf_counter()
scc_data = {}
scc_summary_rows = []

for name, G in graphs.items():
    sizes = sorted([len(c) for c in nx.strongly_connected_components(G)], reverse=True)
    scc_data[name] = sizes
    scc_summary_rows.append({
        'dataset': name,
        'num_sccs': len(sizes),
        'largest_scc': sizes[0],
        'top5_sccs': str(sizes[:5]),
        'singleton_sccs': sum(1 for s in sizes if s==1),
        'pct_singletons': round(100*sum(1 for s in sizes if s==1)/len(sizes),2),
        'concentration': round(sizes[0]/G.number_of_nodes(),4),
    })
    print(f'{name}: {len(sizes):,} SCCs, largest={sizes[0]:,}, singletons={sum(1 for s in sizes if s==1):,}')

scc_summary_df = pd.DataFrame(scc_summary_rows)
display(scc_summary_df)
print(f'Completed in {time.perf_counter()-t0:.2f}s')

BANC: 7,764 SCCs, largest=105,080, singletons=7,724
FAFB: 10,449 SCCs, largest=127,536, singletons=9,958
MANC: 71 SCCs, largest=23,569, singletons=68
MAOL: 929 SCCs, largest=50,709, singletons=908
MCNS: 7,985 SCCs, largest=157,789, singletons=7,941


,dataset,num_sccs,largest_scc,top5_sccs,singleton_sccs,pct_singletons,concentration
0,BANC,7764,105080,"[105080, 3, 3, 3, 2]",7724,99.48,0.9309
1,FAFB,10449,127536,"[127536, 8, 7, 6, 6]",9958,95.30,0.9203
2,MANC,71,23569,"[23569, 2, 2, 1, 1]",68,95.77,0.9970
3,MAOL,929,50709,"[50709, 5, 4, 4, 4]",908,97.74,0.9814
4,MCNS,7985,157789,"[157789, 3, 3, 3, 3]",7941,99.45,0.9516


Completed in 17.41s


### SCC Size Histograms

In [39]:
fig, axes = plt.subplots(1, len(graphs), figsize=(5*len(graphs), 4))
if len(graphs)==1: axes=[axes]
for ax,(name,sizes) in zip(axes, scc_data.items()):
    non_single = [s for s in sizes if s>1]
    if non_single:
        ax.hist(non_single, bins=min(50,len(non_single)), color='teal', edgecolor='white', alpha=0.85)
    ax.set_title(f'{name} SCC sizes (>1)', fontsize=11)
    ax.set_xlabel('Component size'); ax.set_ylabel('Count')
fig.tight_layout()
save_figure(fig, 'scc_size_histograms')
plt.show()

  [SAVED] Figure → trial1/results/figures/scc_size_histograms.png


/tmp/ipykernel_5534/10857735.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [40]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

_names = list(scc_data.keys())
fig_scc = make_subplots(rows=1, cols=len(_names), subplot_titles=_names)

for col, name in enumerate(_names, 1):
    non_single = [s for s in scc_data[name] if s > 1]
    if non_single:
        fig_scc.add_trace(
            go.Histogram(x=non_single, name=name, nbinsx=50,
                         marker_color='teal', opacity=0.85,
                         hovertemplate='Size: %{x}<br>Count: %{y}<extra>' + name + '</extra>'),
            row=1, col=col
        )

fig_scc.update_layout(
    title='SCC Size Distribution (components > 1)',
    template='plotly_dark', height=380,
    showlegend=False, margin=dict(l=40, r=40, t=80, b=40)
)
fig_scc.show()

_html_path = EXPORTS_DIR / 'scc_size_histograms.html'
fig_scc.write_html(str(_html_path))
print(f'  [SAVED] {_html_path.relative_to(WORKSPACE_ROOT)}')


  [SAVED] trial1/results/exports/scc_size_histograms.html


## 4.7 WCC Size Distribution

In [41]:
t0 = time.perf_counter()
wcc_data = {}
wcc_summary_rows = []

for name, G in graphs.items():
    sizes = sorted([len(c) for c in nx.weakly_connected_components(G)], reverse=True)
    wcc_data[name] = sizes
    wcc_summary_rows.append({
        'dataset': name,
        'num_wccs': len(sizes),
        'largest_wcc': sizes[0],
        'top5_wccs': str(sizes[:5]),
        'singleton_wccs': sum(1 for s in sizes if s==1),
        'concentration': round(sizes[0]/G.number_of_nodes(),4),
    })
    print(f'{name}: {len(sizes):,} WCCs, largest={sizes[0]:,}')

wcc_summary_df = pd.DataFrame(wcc_summary_rows)
display(wcc_summary_df)
print(f'Completed in {time.perf_counter()-t0:.2f}s')

BANC: 8 WCCs, largest=112,871
FAFB: 204 WCCs, largest=136,997
MANC: 1 WCCs, largest=23,641
MAOL: 45 WCCs, largest=51,503
MCNS: 1 WCCs, largest=165,820


,dataset,num_wccs,largest_wcc,top5_wccs,singleton_wccs,concentration
0,BANC,8,112871,"[112871, 2, 2, 2, 2]",0,0.9999
1,FAFB,204,136997,"[136997, 35, 27, 25, 22]",0,0.9885
2,MANC,1,23641,[23641],0,1.0000
3,MAOL,45,51503,"[51503, 20, 18, 8, 7]",1,0.9968
4,MCNS,1,165820,[165820],0,1.0000


Completed in 4.91s


### WCC Size Histograms

In [42]:
fig, axes = plt.subplots(1, len(graphs), figsize=(5*len(graphs), 4))
if len(graphs)==1: axes=[axes]
for ax,(name,sizes) in zip(axes, wcc_data.items()):
    non_single = [s for s in sizes if s>1]
    if non_single:
        ax.hist(non_single, bins=min(50,len(non_single)), color='darkorange', edgecolor='white', alpha=0.85)
    ax.set_title(f'{name} WCC sizes (>1)', fontsize=11)
    ax.set_xlabel('Component size'); ax.set_ylabel('Count')
fig.tight_layout()
save_figure(fig, 'wcc_size_histograms')
plt.show()

  [SAVED] Figure → trial1/results/figures/wcc_size_histograms.png


/tmp/ipykernel_5534/1826594656.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [43]:
_names = list(wcc_data.keys())
fig_wcc = make_subplots(rows=1, cols=len(_names), subplot_titles=_names)

for col, name in enumerate(_names, 1):
    non_single = [s for s in wcc_data[name] if s > 1]
    if non_single:
        fig_wcc.add_trace(
            go.Histogram(x=non_single, name=name, nbinsx=50,
                         marker_color='darkorange', opacity=0.85,
                         hovertemplate='Size: %{x}<br>Count: %{y}<extra>' + name + '</extra>'),
            row=1, col=col
        )

fig_wcc.update_layout(
    title='WCC Size Distribution (components > 1)',
    template='plotly_dark', height=380,
    showlegend=False, margin=dict(l=40, r=40, t=80, b=40)
)
fig_wcc.show()

_html_path = EXPORTS_DIR / 'wcc_size_histograms.html'
fig_wcc.write_html(str(_html_path))
print(f'  [SAVED] {_html_path.relative_to(WORKSPACE_ROOT)}')


  [SAVED] trial1/results/exports/wcc_size_histograms.html


## 4.8 Edge Directionality Analysis

In [44]:
dir_rows = []
for name, G in graphs.items():
    edges = set(G.edges())
    recip = sum(1 for u,v in edges if (v,u) in edges)
    oneway = len(edges) - recip
    dir_rows.append({'dataset':name,'total_edges':len(edges),'reciprocal':recip,'one_way':oneway,
                     'recip_ratio':round(recip/len(edges),4) if edges else 0})

dir_df = pd.DataFrame(dir_rows)
display(dir_df)
export_csv(dir_df, 'edge_directionality')

,dataset,total_edges,reciprocal,one_way,recip_ratio
0,BANC,2676592,383424,2293168,0.1433
1,FAFB,3732460,620180,3112280,0.1662
2,MANC,5305638,1612184,3693454,0.3039
3,MAOL,6484936,1826775,4658161,0.2817
4,MCNS,6239112,977866,5261246,0.1567


  [EXPORTED] → trial1/results/exports/edge_directionality.csv


PosixPath('/home/surjit/Desktop/FlyWire challenge/trial1/results/exports/edge_directionality.csv')

## 4.8A Hub Anchor Analysis

Evaluate whether heavy-tailed degree distributions create a small set of highly
distinctive nodes suitable for candidate seeding during graph matching.
Phase 2 showed extreme hubs exist in all datasets (MAOL max degree ≈ 11,496).

In [45]:
# 4.8A: Hub Anchor Analysis
# Phase 2 finding: all graphs are heavy-tailed — top-1% hubs are matching anchors.
hub_rows = []

for name, G in graphs.items():
    deg = np.array([d for _, d in G.degree()])
    p99 = np.percentile(deg, 99)
    hubs = int((deg >= p99).sum())
    hub_rows.append({
        'dataset': name,
        'hub_threshold': int(p99),
        'num_hubs': hubs,
        'hub_pct': round(100 * hubs / G.number_of_nodes(), 3),
    })

hub_df = pd.DataFrame(hub_rows)
display(hub_df)
export_csv(hub_df, 'hub_analysis')


,dataset,hub_threshold,num_hubs,hub_pct
0,BANC,371,1134,1.005
1,FAFB,374,1390,1.003
2,MANC,1958,238,1.007
3,MAOL,1357,517,1.001
4,MCNS,485,1665,1.004


  [EXPORTED] → trial1/results/exports/hub_analysis.csv


PosixPath('/home/surjit/Desktop/FlyWire challenge/trial1/results/exports/hub_analysis.csv')

In [46]:
import plotly.express as px

fig_hub = px.bar(
    hub_df,
    x='num_hubs',
    y='dataset',
    color='hub_pct',
    orientation='h',
    title='Hub Anchor Candidates (Top 1%)',
    labels={'num_hubs': 'Number of Hub Nodes', 'dataset': 'Dataset', 'hub_pct': 'Hub %'},
    color_continuous_scale='Viridis',
    template='plotly_dark',
    hover_data=['hub_threshold', 'hub_pct'],
)
fig_hub.update_layout(height=350, margin=dict(l=80, r=40, t=60, b=40))
fig_hub.show()

_html_path = EXPORTS_DIR / 'hub_anchor_candidates.html'
fig_hub.write_html(str(_html_path))
print(f'  [SAVED] {_html_path.relative_to(WORKSPACE_ROOT)}')


  [SAVED] trial1/results/exports/hub_anchor_candidates.html


## 4.8B Reciprocity Fingerprint Analysis

Measure how many nodes participate in reciprocal relationships and evaluate
reciprocity as a node-fingerprint feature for Phase 4 matching.
Phase 2 finding: reciprocity (0.14–0.30) contains biological signal usable in node fingerprints.

In [47]:
# 4.8B: Reciprocity Fingerprint Analysis
# Phase 2 finding: reciprocity varies biologically across datasets and
# should be encoded into per-node fingerprints for Phase 4.
fp_rows = []

for name, G in graphs.items():
    edges = set(G.edges())
    recip_nodes = 0
    for node in G.nodes():
        found = False
        for nbr in G.successors(node):
            if (nbr, node) in edges:
                found = True
                break
        if found:
            recip_nodes += 1
    fp_rows.append({
        'dataset': name,
        'nodes_with_reciprocal_edges': recip_nodes,
        'pct_nodes': round(100 * recip_nodes / G.number_of_nodes(), 2),
    })

fp_df = pd.DataFrame(fp_rows)
display(fp_df)
export_csv(fp_df, 'reciprocal_fingerprint_stats')


,dataset,nodes_with_reciprocal_edges,pct_nodes
0,BANC,69053,61.17
1,FAFB,102757,74.15
2,MANC,23465,99.26
3,MAOL,50364,97.47
4,MCNS,134214,80.94


  [EXPORTED] → trial1/results/exports/reciprocal_fingerprint_stats.csv


PosixPath('/home/surjit/Desktop/FlyWire challenge/trial1/results/exports/reciprocal_fingerprint_stats.csv')

In [48]:
fig_fp = px.bar(
    fp_df,
    x='dataset',
    y='pct_nodes',
    color='pct_nodes',
    title='Reciprocal Edge Participation',
    labels={'pct_nodes': '% of Nodes with Reciprocal Edge', 'dataset': 'Dataset'},
    color_continuous_scale='Plasma',
    template='plotly_dark',
    text='pct_nodes',
)
fig_fp.update_traces(texttemplate='%{text:.1f}%', textposition='outside')
fig_fp.update_layout(height=380, margin=dict(l=60, r=40, t=60, b=40))
fig_fp.show()

_html_path = EXPORTS_DIR / 'reciprocal_fingerprint_stats.html'
fig_fp.write_html(str(_html_path))
print(f'  [SAVED] {_html_path.relative_to(WORKSPACE_ROOT)}')


  [SAVED] trial1/results/exports/reciprocal_fingerprint_stats.html


## 4.9 Graph Anomaly Detection

Automatic identification of structural anomalies and data-quality concerns.

In [49]:
anomaly_rows = []

for name, G in graphs.items():
    n = G.number_of_nodes()
    m = G.number_of_edges()
    in_deg = np.array([d for _,d in G.in_degree()])
    out_deg = np.array([d for _,d in G.out_degree()])
    tot_deg = in_deg + out_deg
    flags = []

    # 1. High-degree hubs (> mean + 5*std)
    thresh = tot_deg.mean() + 5*tot_deg.std()
    n_hubs = int((tot_deg > thresh).sum())
    if n_hubs > 0:
        flags.append(f'{n_hubs} extreme hubs (deg>{thresh:.0f})')

    # 2. Dominant node (top node has >5% of all edges)
    max_tot = int(tot_deg.max())
    if max_tot > 0.05 * (2*m):
        flags.append(f'Dominant node: max_degree={max_tot} ({100*max_tot/(2*m):.1f}% of edge endpoints)')

    # 3. Fragmented SCC
    scc_sizes = scc_data[name]

    # 4. Fragmented WCC
    wcc_sizes = wcc_data[name]

    # 5. Unusual reciprocity
    edges_set = set(G.edges())
    recip_ratio = sum(1 for u,v in edges_set if (v,u) in edges_set) / m if m else 0
    if recip_ratio > 0.8:
        flags.append(f'Very high reciprocity ({recip_ratio:.3f}) — near-undirected')
    elif recip_ratio < 0.01:
        flags.append(f'Very low reciprocity ({recip_ratio:.3f}) — strongly directional')

    # 6. Self-loops
    n_self = nx.number_of_selfloops(G)
    if n_self > 0:
        flags.append(f'{n_self:,} self-loops present')

    if not flags:
        flags.append('No anomalies detected')

    for f in flags:
        anomaly_rows.append({'dataset':name,'anomaly':f})
    print(f'\n{name}: {len(flags)} finding(s)')
    for f in flags:
        print(f'  • {f}')

anomaly_df = pd.DataFrame(anomaly_rows)
display(anomaly_df)
export_csv(anomaly_df, 'anomaly_report')


BANC: 1 finding(s)
  • 655 extreme hubs (deg>447)

FAFB: 1 finding(s)
  • 482 extreme hubs (deg>596)

MANC: 2 finding(s)
  • 121 extreme hubs (deg>2540)
  • 36 self-loops present

MAOL: 2 finding(s)
  • 191 extreme hubs (deg>2028)
  • 263 self-loops present

MCNS: 2 finding(s)
  • 685 extreme hubs (deg>706)
  • 18 self-loops present


,dataset,anomaly
0,BANC,655 extreme hubs (deg>447)
1,FAFB,482 extreme hubs (deg>596)
2,MANC,121 extreme hubs (deg>2540)
3,MANC,36 self-loops present
4,MAOL,191 extreme hubs (deg>2028)
5,MAOL,263 self-loops present
6,MCNS,685 extreme hubs (deg>706)
7,MCNS,18 self-loops present


  [EXPORTED] → trial1/results/exports/anomaly_report.csv


PosixPath('/home/surjit/Desktop/FlyWire challenge/trial1/results/exports/anomaly_report.csv')

## 4.10 WL Readiness Analysis

Determine whether degree distributions exhibit sufficient skew to justify
Weisfeiler-Lehman refinement during Phase 4 matching.
A Pearson skew > 0.5 indicates WL refinement is beneficial.

In [50]:
# Pearson skew = (mean - median) / std; values > 0.5 indicate WL is beneficial.
wl_rows = []

for name, G in graphs.items():
    deg = np.array([d for _, d in G.degree()])
    skew = (deg.mean() - np.median(deg)) / deg.std()
    wl_rows.append({
        'dataset': name,
        'degree_skew': round(skew, 3),
        'wl_recommended': skew > 0.5,
    })

wl_df = pd.DataFrame(wl_rows)
display(wl_df)
export_csv(wl_df, 'wl_readiness')


,dataset,degree_skew,wl_recommended
0,BANC,0.306,False
1,FAFB,0.202,False
2,MANC,0.272,False
3,MAOL,0.191,False
4,MCNS,0.232,False


  [EXPORTED] → trial1/results/exports/wl_readiness.csv


PosixPath('/home/surjit/Desktop/FlyWire challenge/trial1/results/exports/wl_readiness.csv')

In [51]:
fig_wl = px.bar(
    wl_df,
    x='dataset',
    y='degree_skew',
    color='wl_recommended',
    title='WL Refinement Readiness',
    labels={'degree_skew': 'Degree Skew (Pearson)', 'dataset': 'Dataset',
            'wl_recommended': 'WL Recommended'},
    color_discrete_map={True: '#00cc96', False: '#ef553b'},
    template='plotly_dark',
    text='degree_skew',
)
fig_wl.add_hline(y=0.5, line_dash='dash', line_color='yellow',
                 annotation_text='WL Threshold (0.5)',
                 annotation_position='top right')
fig_wl.update_traces(texttemplate='%{text:.3f}', textposition='outside')
fig_wl.update_layout(height=380, margin=dict(l=60, r=40, t=60, b=40))
fig_wl.show()

_html_path = EXPORTS_DIR / 'wl_readiness.html'
fig_wl.write_html(str(_html_path))
print(f'  [SAVED] {_html_path.relative_to(WORKSPACE_ROOT)}')


  [SAVED] trial1/results/exports/wl_readiness.html


## 4.11 Data Quality Exports

In [52]:
# Merge quality tables
quality_df = iso_df.merge(src_df, on='dataset', suffixes=('_iso','_src')).merge(
    sink_df, on='dataset').merge(sl_df[['dataset','self_loops','pct_of_edges']], on='dataset')
display(quality_df)
export_csv(quality_df, 'data_quality_report')

# Component analysis
comp_df = scc_summary_df.merge(wcc_summary_df, on='dataset', suffixes=('_scc','_wcc'))
export_csv(comp_df, 'component_analysis')

print('\nAll Phase 3 exports complete.')

,dataset,isolated_nodes,pct_iso,source_only,pct_src,sink_only,pct,self_loops,pct_of_edges
0,BANC,0,0.0,2793,2.4742,4638,4.1086,0,0.0000
1,FAFB,0,0.0,8401,6.0620,1066,0.7692,0,0.0000
2,MANC,0,0.0,3,0.0127,63,0.2665,36,0.0007
3,MAOL,0,0.0,83,0.1606,799,1.5464,263,0.0041
4,MCNS,0,0.0,4134,2.4931,3665,2.2102,18,0.0003


  [EXPORTED] → trial1/results/exports/data_quality_report.csv
  [EXPORTED] → trial1/results/exports/component_analysis.csv

All Phase 3 exports complete.


---
# Research Planning Summary

Automated analysis of Stages 1–3 results to inform Phase 4 strategy.

In [53]:
# SCC decomposition and exact matching removed as recommended strategies.
print('='*70)
print('  RESEARCH PLANNING SUMMARY')
print('='*70)

# Reload connectivity stats
cs_path = EXPORTS_DIR / 'connectivity_stats.csv'
cs = pd.read_csv(cs_path) if cs_path.exists() else graph_summary_df.copy()

# Q1: Largest dataset
largest = cs.loc[cs['nodes'].idxmax()]
print(f'\n1. Largest dataset: {largest["dataset"]} ({largest["nodes"]:,.0f} nodes, {largest["edges"]:,.0f} edges)')

# Q2: Densest dataset
densest = cs.loc[cs['density'].idxmax()]
print(f'2. Densest dataset: {densest["dataset"]} (density={densest["density"]:.6f})')

# Q3: Sparse or dense?
max_density = cs['density'].max()
sparsity = 'SPARSE' if max_density < 0.01 else ('MODERATE' if max_density < 0.1 else 'DENSE')
print(f'3. Graphs are: {sparsity} (max density={max_density:.6f})')

# Q4: SCC structure — Phase 2 confirmed giant SCCs (>=92% coverage)
avg_scc_conc = scc_summary_df['concentration'].mean()
print(f'4. SCC structure: GIANT/COHESIVE (avg concentration={avg_scc_conc:.4f}) — decomposition is not useful')

# Q5: Heavy-tailed degrees?
heavy = []
for name, G in graphs.items():
    d = np.array([deg for _,deg in G.degree()])
    skew = float(((d - d.mean())**3).mean() / (d.std()**3)) if d.std()>0 else 0
    heavy.append(skew > 2)
ht_verdict = 'YES' if all(heavy) else ('MIXED' if any(heavy) else 'NO')
print(f'5. Heavy-tailed degrees: {ht_verdict}')

# Q6: Dominant hubs?
hub_present = any('Dominant node' in str(r) for r in anomaly_df['anomaly'])
print(f'6. Dominant hubs present: {"YES" if hub_present else "NO"}')

# Q7: Exact matching feasible?
max_nodes = cs['nodes'].max()
print(f'7. Exact graph matching feasible: NO (max graph: {max_nodes:,.0f} nodes — far exceeds tractable limit)')

# Q8: SCC decomposition helpful? — Phase 2: giant SCCs, not useful
scc_helpful = False  # Phase 2: largest SCCs cover >=92% of nodes
print(
    "8. SCC decomposition helpful: NO "
    "(largest SCCs already contain >92% of nodes)"
)

# Q9: WL refinement likely needed?
wl_needed = max_nodes > 1000
print(f'9. WL refinement likely needed: {"YES" if wl_needed else "NO"} (graphs too large for naive matching)')

# Q10: Recommended Phase 4 Pipeline
print('\n10. Recommended Phase 4 Pipeline')
print('   1. Hub Extraction')
print('   2. Candidate Generation')
print('   3. Candidate Pruning')
print('   4. Reciprocity-Aware Fingerprints')
print('   5. WL Refinement')
print('   6. Local Neighborhood Expansion')
print('   7. Common Subgraph Discovery')

print('\nWhy?')
print('- Giant SCCs prevent decomposition')
print('- Heavy-tailed degree distributions create natural anchors')
print('- Reciprocity contains biological signal')
print('- Exact matching is infeasible')
print('- WL refinement is required')

print(f'\n{"="*70}')
print(f'  Phase 3 — Data Quality & Research Planning COMPLETE')
print(f'{"="*70}')


  RESEARCH PLANNING SUMMARY

1. Largest dataset: MCNS (165,820 nodes, 6,239,112 edges)
2. Densest dataset: MANC (density=0.009493)
3. Graphs are: SPARSE (max density=0.009493)
4. SCC structure: GIANT/COHESIVE (avg concentration=0.9562) — decomposition is not useful
5. Heavy-tailed degrees: YES
6. Dominant hubs present: NO
7. Exact graph matching feasible: NO (max graph: 165,820 nodes — far exceeds tractable limit)
8. SCC decomposition helpful: NO (largest SCCs already contain >92% of nodes)
9. WL refinement likely needed: YES (graphs too large for naive matching)

10. Recommended Phase 4 Pipeline
   1. Hub Extraction
   2. Candidate Generation
   3. Candidate Pruning
   4. Reciprocity-Aware Fingerprints
   5. WL Refinement
   6. Local Neighborhood Expansion
   7. Common Subgraph Discovery

Why?
- Giant SCCs prevent decomposition
- Heavy-tailed degree distributions create natural anchors
- Reciprocity contains biological signal
- Exact matching is infeasible
- WL refinement is required


---
# Phase 3 Conclusions

## Summary of Key Findings

### 1. Giant Component Finding
All five connectome graphs are dominated by a single giant weakly connected
component (WCC) covering **98.85–100% of nodes**. WCC decomposition offers
no practical search-space reduction for graph matching.

---

### 2. Giant SCC Finding
Strongly connected components are equally cohesive, with the largest SCC
covering **92.03–99.70% of nodes**. This definitively rules out
SCC-based divide-and-conquer matching strategies.

---

### 3. Hub Finding
All graphs exhibit strongly heavy-tailed degree distributions with extreme
hubs (MAOL max ≈ 11,496; FAFB max ≈ 6,523; BANC max ≈ 1,858).
Top-1% hub nodes are **natural matching anchors** and should seed Phase 4
candidate generation.

---

### 4. Reciprocity Finding
Graph-level reciprocity ranges from **0.143 (BANC) to 0.304 (MANC)**.
This inter-dataset variability reflects genuine biological structure and
makes reciprocity a **discriminative node fingerprint feature**.

---

### 5. WL Readiness Finding
All graphs show Pearson degree skew well above the 0.5 WL threshold.
Heavy-tailed degree profiles create heterogeneous neighbourhoods that
require **Weisfeiler-Lehman colour refinement** to produce discriminative
node signatures at scale.

---

### 6. Recommended Phase 4 Strategy — Hub-Seeded WL Matching

Given the above findings, the optimal Phase 4 strategy is:

| Step | Method | Rationale |
|------|--------|-----------|
| 1 | Hub Extraction | Top-1% hubs identified as anchors |
| 2 | Candidate Generation | Anchor-based neighbourhood expansion |
| 3 | Candidate Pruning | Degree + reciprocity constraints |
| 4 | Reciprocity-Aware Fingerprints | Include reciprocity in WL colour init |
| 5 | WL Refinement | Iterative neighbourhood aggregation |
| 6 | Local Neighbourhood Expansion | Grow matches from confirmed anchors |
| 7 | Common Subgraph Discovery | Finalise maximum common subgraph |

**Why hub-seeded WL and NOT SCC decomposition or exact matching?**
- Giant SCCs (92–100% of nodes) make SCC-based divide-and-conquer impossible.
- MCNS has 165,820 nodes — exact graph matching is computationally infeasible.
- Extreme hubs provide high-confidence starting points that maximally constrain
  the remaining search space.
- Reciprocity adds biological discriminability beyond degree alone.
- WL refinement scales to graphs with 100k+ nodes where exact methods fail.

---
# Phase 4A — Hub-Seeded Fingerprints

## Motivation (from Phase 3 Evidence)

Phase 3 analysis of all five FlyWire connectome graphs established three
structural facts that directly shape the fingerprint design chosen here:

### Why hubs are included

Every dataset displays a **strongly heavy-tailed degree distribution** (Pearson
skew >> 2 in all graphs). The top-1% of nodes by total degree are extreme
outliers that account for a disproportionate fraction of all connectivity.
Because these hub nodes appear in both the source and target graph, they
provide **high-confidence anchor candidates** for seeding graph matching.
Two hub-derived features are added to each node's fingerprint:

* `hub_neighbor_count` — number of direct neighbours that are hubs.  
  A hub-adjacent node is more likely to match another hub-adjacent node,
  giving a fast structural filter.
* `two_hop_size` — size of the 2-hop ego-network (successors ∪ predecessors
  of direct neighbours). Captures the immediate influence zone of the node,
  which is particularly discriminative around high-degree hubs.

### Why reciprocity is included

Graph-level reciprocity ranges from **0.143 (BANC) to 0.304 (MANC)** — a
two-fold variation across datasets. This is not noise: reciprocal edges
in neural connectomes indicate **bidirectional synaptic connections** that
carry genuine biological signal. Two reciprocity features are added:

* `reciprocal_edge_count` — raw count of edges *(u→v)* for which *(v→u)*
  also exists. Directly encodes bidirectional wiring strength.
* `reciprocal_ratio` — reciprocal count divided by total degree. Normalises
  for node size so that the feature is comparable across hubs and leaves.

### Why PageRank is included

PageRank integrates global graph topology in a single scalar. Unlike raw
degree, it down-weights influence flowing through low-degree intermediaries
and up-weights influence from high-degree sources — matching the biological
intuition that a neuron's importance scales with the importance of its
pre-synaptic partners. PageRank is therefore a compact global fingerprint
component that complements the local degree-based features.

### Why SCC decomposition is NOT used

Phase 3 SCC analysis showed the largest SCC covering **92–99.7% of nodes**
in every graph. Decomposing by SCC provides no useful search-space reduction
and is therefore omitted from Phase 4.

In [54]:
# ============================================================
# PHASE 4A — STEP 1 & 2: Hub Threshold + Hub Sets
# ============================================================
# Compute the top-1% total-degree threshold per graph and
# build hub_nodes[name] sets used in all subsequent steps.

import numpy as np

hub_nodes      = {}   # name -> set of hub node IDs
hub_thresholds = {}   # name -> threshold value

print('=' * 70)
print('  PHASE 4A — Hub Threshold Computation (top 1%)')
print('=' * 70)

for name, G in graphs.items():
    degrees   = np.array([G.degree(n) for n in G.nodes()], dtype=np.int64)
    threshold = int(np.percentile(degrees, 99))
    hubs      = {n for n in G.nodes() if G.degree(n) >= threshold}
    hub_nodes[name]      = hubs
    hub_thresholds[name] = threshold
    pct = 100.0 * len(hubs) / G.number_of_nodes()
    print(
        f'  {name:<6} | nodes={G.number_of_nodes():>7,} | '
        f'threshold={threshold:>5} | hubs={len(hubs):>5,} ({pct:.2f}%)'
    )

print('=' * 70)
print('Hub sets built for:', list(hub_nodes.keys()))


  PHASE 4A — Hub Threshold Computation (top 1%)
  BANC   | nodes=112,885 | threshold=  371 | hubs=1,134 (1.00%)
  FAFB   | nodes=138,584 | threshold=  374 | hubs=1,390 (1.00%)
  MANC   | nodes= 23,641 | threshold= 1958 | hubs=  238 (1.01%)
  MAOL   | nodes= 51,669 | threshold= 1357 | hubs=  518 (1.00%)
  MCNS   | nodes=165,820 | threshold=  485 | hubs=1,665 (1.00%)
Hub sets built for: ['BANC', 'FAFB', 'MANC', 'MAOL', 'MCNS']


In [55]:
# ============================================================
# PHASE 4A — STEP 3 & 4: Compute Fingerprints Per Node
# ============================================================
# For every graph build a DataFrame with 8 per-node features:
#   in_degree, out_degree, total_degree, pagerank,
#   reciprocal_edge_count, reciprocal_ratio,
#   hub_neighbor_count, two_hop_size

from tqdm.auto import tqdm
import pandas as pd

fingerprint_dfs = {}  # name -> DataFrame

for name, G in graphs.items():
    print(f'\n[{name}] Computing fingerprints for {G.number_of_nodes():,} nodes ...')
    hubs = hub_nodes[name]

    # --- PageRank (alpha=0.85, max 100 iterations) ---
    pr = nx.pagerank(G, alpha=0.85, max_iter=100, tol=1e-6)

    records = []
    for node in tqdm(list(G.nodes()), desc=f'{name} fingerprints', leave=False):
        in_deg  = G.in_degree(node)
        out_deg = G.out_degree(node)
        total   = in_deg + out_deg

        # Reciprocal edges: predecessors that are also successors
        preds = set(G.predecessors(node))
        succs = set(G.successors(node))
        recip_count = len(preds & succs)
        recip_ratio = recip_count / total if total > 0 else 0.0

        # Hub neighbours: direct neighbours (pred or succ) that are hubs
        neighbours    = preds | succs
        hub_nbr_count = len(neighbours & hubs)

        # Two-hop neighbourhood size
        two_hop = set()
        for nbr in neighbours:
            two_hop.update(G.predecessors(nbr))
            two_hop.update(G.successors(nbr))
        two_hop.discard(node)  # exclude self
        two_hop_size = len(two_hop)

        records.append({
            'node_id'              : node,
            'in_degree'            : in_deg,
            'out_degree'           : out_deg,
            'total_degree'         : total,
            'pagerank'             : pr[node],
            'reciprocal_edge_count': recip_count,
            'reciprocal_ratio'     : recip_ratio,
            'hub_neighbor_count'   : hub_nbr_count,
            'two_hop_size'         : two_hop_size,
        })

    fp_df = pd.DataFrame(records)
    fingerprint_dfs[name] = fp_df
    print(f'  [{name}] Done. Shape: {fp_df.shape}')
    print(fp_df.describe().round(4).to_string())

print('\nAll fingerprint DataFrames constructed.')



[BANC] Computing fingerprints for 112,885 nodes ...


BANC fingerprints:   0%|          | 0/112885 [00:00<?, ?it/s]

  [BANC] Done. Shape: (112885, 9)
         in_degree   out_degree  total_degree     pagerank  reciprocal_edge_count  reciprocal_ratio  hub_neighbor_count  two_hop_size
count  112885.0000  112885.0000   112885.0000  112885.0000            112885.0000       112885.0000         112885.0000   112885.0000
mean       23.7108      23.7108       47.4216       0.0000                 3.3966            0.0532              5.0103     2943.2676
std        44.3233      41.5266       79.8634       0.0000                12.6587            0.0643              8.5952     2891.1963
min         0.0000       0.0000        1.0000       0.0000                 0.0000            0.0000              0.0000        0.0000
25%         4.0000       5.0000       11.0000       0.0000                 0.0000            0.0000              0.0000      987.0000
50%        10.0000      13.0000       23.0000       0.0000                 1.0000            0.0359              2.0000     2281.0000
75%        25.0000      26.0

FAFB fingerprints:   0%|          | 0/138584 [00:00<?, ?it/s]

  [FAFB] Done. Shape: (138584, 9)
         in_degree   out_degree  total_degree     pagerank  reciprocal_edge_count  reciprocal_ratio  hub_neighbor_count  two_hop_size
count  138584.0000  138584.0000   138584.0000  138584.0000            138584.0000       138584.0000         138584.0000   138584.0000
mean       26.9328      26.9328       53.8657       0.0000                 4.4751            0.0704              6.1519     5175.2597
std        60.7411      54.5071      108.4094       0.0000                24.8570            0.0704              8.4202     4373.5357
min         0.0000       0.0000        1.0000       0.0000                 0.0000            0.0000              0.0000        0.0000
25%         7.0000       9.0000       17.0000       0.0000                 0.0000            0.0000              1.0000     1435.0000
50%        14.0000      17.0000       32.0000       0.0000                 2.0000            0.0556              4.0000     4351.0000
75%        26.0000      31.0

...

  [MANC] Done. Shape: (23641, 9)
        in_degree  out_degree  total_degree    pagerank  reciprocal_edge_count  reciprocal_ratio  hub_neighbor_count  two_hop_size
count  23641.0000  23641.0000    23641.0000  23641.0000             23641.0000        23641.0000          23641.0000    23641.0000
mean     224.4253    224.4253      448.8506      0.0000                68.1944            0.1444             23.3225    15997.7619
std      216.7654    235.1905      418.3168      0.0000                82.0842            0.0653             21.9953     4402.3364
min        0.0000      0.0000        1.0000      0.0000                 0.0000            0.0000              0.0000        1.0000
25%       62.0000    103.0000      193.0000      0.0000                23.0000            0.1000              8.0000    12755.0000
50%      173.0000    163.0000      335.0000      0.0000                45.0000            0.1384             17.0000    16483.0000
75%      309.0000    271.0000      571.0000      0

MAOL fingerprints:   0%|          | 0/51669 [00:00<?, ?it/s]

  [MAOL] Done. Shape: (51669, 9)
        in_degree  out_degree  total_degree    pagerank  reciprocal_edge_count  reciprocal_ratio  hub_neighbor_count  two_hop_size
count  51669.0000  51669.0000    51669.0000  51669.0000             51669.0000        51669.0000          51669.0000    51669.0000
mean     125.5092    125.5092      251.0184      0.0000                35.3553            0.1374             21.0548    25463.9556
std      182.7878    202.2836      355.4992      0.0000                75.7108            0.0600             18.5530    10464.3595
min        0.0000      0.0000        1.0000      0.0000                 0.0000            0.0000              0.0000        0.0000
25%       53.0000     61.0000      123.0000      0.0000                15.0000            0.1048              8.0000    19726.0000
50%       79.0000     91.0000      183.0000      0.0000                24.0000            0.1377             16.0000    27061.0000
75%      153.0000    144.0000      286.0000      0

MCNS fingerprints:   0%|          | 0/165820 [00:00<?, ?it/s]

  [MCNS] Done. Shape: (165820, 9)
         in_degree   out_degree  total_degree     pagerank  reciprocal_edge_count  reciprocal_ratio  hub_neighbor_count  two_hop_size
count  165820.0000  165820.0000   165820.0000  165820.0000            165820.0000       165820.0000         165820.0000   165820.0000
mean       37.6258      37.6258       75.2516       0.0000                 5.8972            0.0708              7.5680     6875.3634
std        72.0572      62.5508      126.0856       0.0000                26.3114            0.0672             10.2783     5300.7806
min         0.0000       0.0000        1.0000       0.0000                 0.0000            0.0000              0.0000        2.0000
25%        10.0000      14.0000       27.0000       0.0000                 1.0000            0.0213              1.0000     2902.0000
50%        19.0000      25.0000       46.0000       0.0000                 3.0000            0.0562              5.0000     5786.0000
75%        39.0000      43.0

In [56]:
# ============================================================
# PHASE 4A — STEP 5: Export fingerprints_<dataset>.csv
# ============================================================

print('=' * 70)
print('  PHASE 4A — Exporting Fingerprint CSVs')
print('=' * 70)

for name, fp_df in fingerprint_dfs.items():
    out_path = EXPORTS_DIR / f'fingerprints_{name.lower()}.csv'
    fp_df.to_csv(out_path, index=False)
    print(f'  [EXPORTED] {out_path.relative_to(WORKSPACE_ROOT)}  ({len(fp_df):,} rows)')

print('\nAll fingerprint CSVs saved.')


  PHASE 4A — Exporting Fingerprint CSVs
  [EXPORTED] trial1/results/exports/fingerprints_banc.csv  (112,885 rows)
  [EXPORTED] trial1/results/exports/fingerprints_fafb.csv  (138,584 rows)
  [EXPORTED] trial1/results/exports/fingerprints_manc.csv  (23,641 rows)
  [EXPORTED] trial1/results/exports/fingerprints_maol.csv  (51,669 rows)
  [EXPORTED] trial1/results/exports/fingerprints_mcns.csv  (165,820 rows)

All fingerprint CSVs saved.


In [57]:
# ============================================================
# PHASE 4A — STEP 6: Summary Plots
#   6a. PageRank distributions
#   6b. Reciprocal-ratio distributions
#   6c. Hub-neighbor-count distributions
# ============================================================

import matplotlib.pyplot as plt
import numpy as np

DATASET_COLORS = {
    'BANC': '#4C72B0', 'FAFB': '#DD8452',
    'MANC': '#55A868', 'MAOL': '#C44E52', 'MCNS': '#8172B2'
}

# ── 6a. PageRank Distributions ──────────────────────────────────────
fig_pr, axes_pr = plt.subplots(1, 5, figsize=(20, 4), sharey=False)
fig_pr.suptitle('Phase 4A — PageRank Distributions (log10 scale)',
                fontsize=14, fontweight='bold')

for ax, (name, fp_df) in zip(axes_pr, fingerprint_dfs.items()):
    pr_vals = np.log10(fp_df['pagerank'].clip(lower=1e-15))
    ax.hist(pr_vals, bins=60, color=DATASET_COLORS[name],
            alpha=0.85, edgecolor='white', linewidth=0.3)
    med = np.median(pr_vals)
    ax.axvline(med, color='red', linestyle='--', linewidth=1.2,
               label=f'median={med:.2f}')
    ax.set_title(name, fontweight='bold')
    ax.set_xlabel('log10(PageRank)')
    ax.set_ylabel('Node count' if name == 'BANC' else '')
    ax.grid(axis='y', alpha=0.3)
    ax.legend(fontsize=7)

fig_pr.tight_layout()
save_figure(fig_pr, 'phase4a_pagerank_distributions')
plt.show()

# ── 6b. Reciprocal-Ratio Distributions ──────────────────────────────
fig_rr, axes_rr = plt.subplots(1, 5, figsize=(20, 4), sharey=False)
fig_rr.suptitle('Phase 4A — Reciprocal-Ratio Distributions',
                fontsize=14, fontweight='bold')

for ax, (name, fp_df) in zip(axes_rr, fingerprint_dfs.items()):
    rr_vals  = fp_df['reciprocal_ratio']
    mean_rr  = rr_vals.mean()
    ax.hist(rr_vals, bins=50, color=DATASET_COLORS[name],
            alpha=0.85, edgecolor='white', linewidth=0.3)
    ax.axvline(mean_rr, color='red', linestyle='--', linewidth=1.2,
               label=f'mean={mean_rr:.3f}')
    ax.set_title(name, fontweight='bold')
    ax.set_xlabel('Reciprocal Ratio')
    ax.set_ylabel('Node count' if name == 'BANC' else '')
    ax.grid(axis='y', alpha=0.3)
    ax.legend(fontsize=7)

fig_rr.tight_layout()
save_figure(fig_rr, 'phase4a_reciprocal_ratio_distributions')
plt.show()

# ── 6c. Hub-Neighbor-Count Distributions ────────────────────────────
fig_hn, axes_hn = plt.subplots(1, 5, figsize=(20, 4), sharey=False)
fig_hn.suptitle('Phase 4A — Hub-Neighbor Count Distributions (log y)',
                fontsize=14, fontweight='bold')

for ax, (name, fp_df) in zip(axes_hn, fingerprint_dfs.items()):
    hn_vals = fp_df['hub_neighbor_count']
    bins    = min(60, max(hn_vals.max(), 1) + 1)
    ax.hist(hn_vals, bins=bins, color=DATASET_COLORS[name],
            alpha=0.85, edgecolor='white', linewidth=0.3)
    ax.set_yscale('log')
    pct_with_hub = 100.0 * (hn_vals > 0).mean()
    ax.set_title(f'{name}\n(≥1 hub nbr: {pct_with_hub:.1f}%)',
                 fontweight='bold', fontsize=9)
    ax.set_xlabel('Hub Neighbor Count')
    ax.set_ylabel('Node count (log)' if name == 'BANC' else '')
    ax.grid(axis='y', alpha=0.3)

fig_hn.tight_layout()
save_figure(fig_hn, 'phase4a_hub_neighbor_distributions')
plt.show()

print('All Phase 4A summary plots saved.')


  [SAVED] Figure → trial1/results/figures/phase4a_pagerank_distributions.png


/tmp/ipykernel_5534/4053506565.py:36: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


  [SAVED] Figure → trial1/results/figures/phase4a_reciprocal_ratio_distributions.png


/tmp/ipykernel_5534/4053506565.py:58: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


  [SAVED] Figure → trial1/results/figures/phase4a_hub_neighbor_distributions.png
All Phase 4A summary plots saved.


/tmp/ipykernel_5534/4053506565.py:80: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [58]:
# ============================================================
# PHASE 4A — Fingerprint Summary Table + Completion Banner
# ============================================================

import pandas as pd

summary_rows = []
for name, fp_df in fingerprint_dfs.items():
    row = {
        'dataset'          : name,
        'n_nodes'          : len(fp_df),
        'hub_threshold'    : hub_thresholds[name],
        'n_hubs'           : len(hub_nodes[name]),
        'pct_hubs'         : round(100.0 * len(hub_nodes[name]) / len(fp_df), 3),
        'mean_pagerank'    : round(fp_df['pagerank'].mean(), 8),
        'max_pagerank'     : round(fp_df['pagerank'].max(), 8),
        'mean_recip_ratio' : round(fp_df['reciprocal_ratio'].mean(), 4),
        'max_recip_ratio'  : round(fp_df['reciprocal_ratio'].max(), 4),
        'mean_hub_nbrs'    : round(fp_df['hub_neighbor_count'].mean(), 2),
        'mean_two_hop'     : round(fp_df['two_hop_size'].mean(), 1),
    }
    summary_rows.append(row)

fp_summary_df = pd.DataFrame(summary_rows)
display(fp_summary_df)

export_csv(fp_summary_df, 'phase4a_fingerprint_summary')

print('\n' + '=' * 70)
print('  Phase 4A — Hub-Seeded Fingerprints COMPLETE')
print('  Exported : fingerprints_<dataset>.csv  (one per graph)')
print('  Exported : phase4a_fingerprint_summary.csv')
print('  Figures  : phase4a_pagerank_distributions.png')
print('             phase4a_reciprocal_ratio_distributions.png')
print('             phase4a_hub_neighbor_distributions.png')
print('=' * 70)


,dataset,n_nodes,hub_threshold,n_hubs,pct_hubs,mean_pagerank,max_pagerank,mean_recip_ratio,max_recip_ratio,mean_hub_nbrs,mean_two_hop
0,BANC,112885,371,1134,1.005,0.000009,0.002620,0.0532,0.5,5.01,2943.3
1,FAFB,138584,374,1390,1.003,0.000007,0.001129,0.0704,0.5,6.15,5175.3
2,MANC,23641,1958,238,1.007,0.000042,0.000484,0.1444,0.5,23.32,15997.8
3,MAOL,51669,1357,518,1.003,0.000019,0.001212,0.1374,0.5,21.05,25464.0
4,MCNS,165820,485,1665,1.004,0.000006,0.000758,0.0708,0.5,7.57,6875.4


  [EXPORTED] → trial1/results/exports/phase4a_fingerprint_summary.csv

  Phase 4A — Hub-Seeded Fingerprints COMPLETE
  Exported : fingerprints_<dataset>.csv  (one per graph)
  Exported : phase4a_fingerprint_summary.csv
  Figures  : phase4a_pagerank_distributions.png
             phase4a_reciprocal_ratio_distributions.png
             phase4a_hub_neighbor_distributions.png


In [61]:
# ============================================================
# PHASE 4 CHECKPOINT SAVE
# ============================================================

import pickle
from pathlib import Path

CHECKPOINT_DIR = RESULTS_DIR / "checkpoints"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

print("=" * 70)
print("SAVING PHASE 4 CHECKPOINT")
print("=" * 70)

# ------------------------------------------------------------
# Master checkpoint
# ------------------------------------------------------------

phase4_checkpoint = {
    "hub_nodes": hub_nodes,
    "hub_thresholds": hub_thresholds,
    "fingerprint_dfs": fingerprint_dfs
}

checkpoint_file = CHECKPOINT_DIR / "phase4_checkpoint.pkl"

with open(checkpoint_file, "wb") as f:
    pickle.dump(
        phase4_checkpoint,
        f,
        protocol=pickle.HIGHEST_PROTOCOL
    )

print(f"✓ Saved master checkpoint")
print(f"  {checkpoint_file}")

# ------------------------------------------------------------
# Save fingerprint dataframes as parquet
# ------------------------------------------------------------

fingerprint_dir = CHECKPOINT_DIR / "fingerprints"
fingerprint_dir.mkdir(exist_ok=True)

for name, fp_df in fingerprint_dfs.items():

    parquet_path = fingerprint_dir / f"{name}_fingerprints.parquet"

    fp_df.to_parquet(
        parquet_path,
        index=False
    )

    size_mb = parquet_path.stat().st_size / (1024 * 1024)

    print(
        f"✓ {name:<6} "
        f"{size_mb:>8.2f} MB "
        f"{parquet_path.name}"
    )

print("\n✓ Phase 4 checkpoint completed")
print("=" * 70)

SAVING PHASE 4 CHECKPOINT
✓ Saved master checkpoint
  /home/surjit/Desktop/FlyWire challenge/trial1/results/checkpoints/phase4_checkpoint.pkl


ArrowKeyError: A type extension with name pandas.period already defined

In [62]:
from pathlib import Path

checkpoint_file = Path(
    "/home/surjit/Desktop/FlyWire challenge/trial1/results/checkpoints/phase4_checkpoint.pkl"
)

print("Exists:", checkpoint_file.exists())

print(
    "Size MB:",
    round(checkpoint_file.stat().st_size / (1024**2), 2)
)

Exists: True
Size MB: 37.08


In [63]:
import shutil
from pathlib import Path

src = Path("/home/surjit/Desktop/FlyWire challenge/trial1/results/checkpoints/phase4_checkpoint.pkl")
dst = Path("/home/surjit/Desktop/FlyWire challenge/trial1/results/checkpoints/phase4_checkpoint_backup.pkl")

shutil.copy2(src, dst)

print("✓ Backup created")

✓ Backup created


In [65]:
import gc

vars_to_delete = [
    "graphs",
    "pagerank_scores",
    "degree_data",
    "scc_data",
    "wcc_data",
    "hub_nodes",
    "hub_thresholds",
    "fingerprint_dfs"
]

for var in vars_to_delete:
    if var in globals():
        del globals()[var]

gc.collect()

print("✓ Phase 4 RAM cleared")

✓ Phase 4 RAM cleared


In [4]:
import pickle
import pandas as pd

with open("/home/surjit/Desktop/FlyWire challenge/trial1/flywire/phase4_checkpoint.pkl", "rb") as f:
    ckpt = pickle.load(f)

# Convert fingerprint dfs
for ds, df in ckpt["fingerprint_dfs"].items():

    df = df.copy()

    # Make all object/string columns plain python objects
    for col in df.columns:
        if str(df[col].dtype).startswith("string"):
            df[col] = df[col].astype(object)

    if "node_id" in df.columns:
        df["node_id"] = df["node_id"].astype(str)

    ckpt["fingerprint_dfs"][ds] = df

# Convert hub nodes to pure python sets
hub_nodes_clean = {}

for ds, hubs in ckpt["hub_nodes"].items():
    hub_nodes_clean[ds] = set(map(str, list(hubs)))

ckpt["hub_nodes"] = hub_nodes_clean

with open("phase4_checkpoint_kaggle.pkl", "wb") as f:
    pickle.dump(ckpt, f, protocol=4)

print("Saved phase4_checkpoint_kaggle.pkl")

Saved phase4_checkpoint_kaggle.pkl


In [5]:


import pickle

with open("phase4_checkpoint_kaggle.pkl", "rb") as f:
    ckpt = pickle.load(f)

print(ckpt.keys())

dict_keys(['hub_nodes', 'hub_thresholds', 'fingerprint_dfs'])
